Steps:
1. Load raw CSV
2. Type conversions & trip end filling
3. Rename to snake_case
4. Impute lat/lon from census tract
5. Impute census tract from lat/lon (lookup)
6. Drop rows with no spatial data
7. Payment type cleanup
8. Company standardization
9. Duplicate check
10. Remove invalid trips (any zero/below-threshold)
11. Consistency & logic checks
12. Absolute outlier filtering
13. Save cleaned parquet
14. ID remapping + compact parquet
15. Time features from trip_start
16. Cyclic encoding (sin/cos)
17. Electronic payment fee detection + save compact parquet
18. Save removed rows

To run this notebook it is necessary to have the Taxi_Trip data downloaded as csv and names as "<i>Taxi_Trips_(2024-).csv.</i>"

## 0) Setup and data loading
First all necessary packages are imported, paths are setup and data is then loaded.

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import os
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = Path(os.getcwd()).parent
DATA_DIR = PROJECT / "data"

CSV_PATH = DATA_DIR / "Taxi_Trips_(2024-).csv"
OUTPUT_CLEAN = DATA_DIR / "Taxi_Trips_cleaned.parquet"
OUTPUT_COMPACT = DATA_DIR / "Taxi_Trips_compact.parquet"
OUTPUT_TRIP_ID_MAP = DATA_DIR / "trip_id_mapping.csv"
OUTPUT_TAXI_ID_MAP = DATA_DIR / "taxi_id_mapping.csv"
OUTPUT_REMOVED = DATA_DIR / "Removed_Rows.parquet"
OUTPUT_NO_SPATIAL = DATA_DIR / "No_Spatial_Value.parquet"
OUTPUT_COMPANIES = DATA_DIR / "company_names_after_cleaning.csv"

removed_frames = []

def flag_and_remove(df, mask, reason):
    n = mask.sum()
    if n == 0:
        return df
    removed = df.loc[mask].copy()
    removed["removal_reason"] = reason
    removed_frames.append(removed)
    print(f"  Removed {n:,} rows: {reason}")
    return df[~mask].reset_index(drop=True)

print("Setup complete.")
print(f"Project: {PROJECT}")
print(f"Data dir: {DATA_DIR}")



In [ ]:
data = pl.read_csv(CSV_PATH, low_memory=False)
original_rows = data.shape[0]
statistics = [("Original row count", original_rows)]
print(f"Loaded {data.shape[0]:,} rows, {data.shape[1]} columns")
print(f'DATA_DIR: {data.dtypes}')

print('Example columns:')
display(data.head(2))
data = data.lazy()

# 1) Type casting and renaming
Next step is to convert the types to the correct data types as we can see above that not all columns are numeric.

In [ ]:
# Casting
money_cols = ["Fare", "Tips", "Tolls", "Extras",  "Trip Total"]

data = data.with_columns(
    pl.col(money_cols)
    .str.replace("$", "", literal=True)
    .str.replace(",", ".", literal=True)
    .cast(pl.Float64, strict=False),
).with_columns( # Miles to float
    pl.col('Trip Miles').str.replace(",", ".", literal=True).cast(pl.Float64, strict=False)
).with_columns( # timestamp casting
    pl.col("Trip Start Timestamp").str.to_datetime("%m/%d/%Y %I:%M:%S %p").alias("Trip Start Timestamp"),
    pl.col("Trip End Timestamp").str.to_datetime("%m/%d/%Y %I:%M:%S %p").alias("Trip End Timestamp")
). with_columns([
    pl.col("Pickup Centroid Latitude").str.replace(",", ".").cast(pl.Float64),
    pl.col("Pickup Centroid Longitude").str.replace(",", ".").cast(pl.Float64),
    pl.col("Dropoff Centroid Latitude").str.replace(",", ".").cast(pl.Float64),
    pl.col("Dropoff Centroid Longitude").str.replace(",", ".").cast(pl.Float64)]
).with_columns(
    pl.col("Pickup Census Tract").cast(pl.Int64, strict=False),
    pl.col("Dropoff Census Tract").cast(pl.Int64, strict=False)
)
data = data.collect()
display(data.head(2))

In [ ]:
snake_map = {
    "Trip ID": "trip_id",
    "Taxi ID": "taxi_id",
    "Trip Start Timestamp": "trip_start",
    "Trip End Timestamp": "trip_end",
    "Trip Seconds": "trip_seconds",
    "Trip Miles": "trip_miles",
    "Pickup Census Tract": "pickup_census_tract",
    "Dropoff Census Tract": "dropoff_census_tract",
    "Pickup Community Area": "pickup_community_area",
    "Dropoff Community Area": "dropoff_community_area",
    "Fare": "fare_usd",
    "Tips": "tips_usd",
    "Tolls": "tolls_usd",
    "Extras": "extras_usd",
    "Trip Total": "trip_total_usd",
    "Payment Type": "payment_type",
    "Company": "company",
    "Pickup Centroid Latitude": "pickup_lat",
    "Pickup Centroid Longitude": "pickup_lon",
    "Pickup Centroid Location": "pickup_location",
    "Dropoff Centroid Latitude": "dropoff_lat",
    "Dropoff Centroid Longitude": "dropoff_lon",
    "Dropoff Centroid  Location": "dropoff_location",
}
# Polars: returns a new DataFrame, so reassign
data = data.rename(snake_map)

print(f"Renamed {len(snake_map)} columns.")
print(f'New column names: {data.columns}')

# 3) Missing values
In this chapter missing values are handled.

In [ ]:
missing_counts = data.select([
    pl.col(col).is_null().sum().alias(col) for col in data.columns
])
print("Missing values by column:")
display(missing_counts)

## 3.1) Missing time values
Missing time values can appear in trip_start, trip_end and trip_seconds. We can see that the start time is always available but trip_end and trip seconds are incomplete.

To fill the missing trip_end we use the start and seconds of the trip to estimate the end. In this case there are non which can be filled like that.

In [ ]:
fill_condition = (
    pl.col("trip_end").is_null() &
    pl.col("trip_start").is_not_null() &
    pl.col("trip_seconds").is_not_null()
)
print(f"Number of rows that will be affected: {data.filter(fill_condition).shape[0]}")  

data = data.with_columns([
    pl.when(fill_condition)
    .then(pl.col("trip_start") + pl.duration(seconds=pl.col("trip_seconds")))
    .otherwise(pl.col("trip_end"))
    .alias("trip_end")
])

To fill missing trip_seconds we can use the start and end to calculate the duration.

In [ ]:
# fill missing trip_seconds where possible
fill_condition_seconds = (
    pl.col("trip_seconds").is_null() &
    pl.col("trip_start").is_not_null() &
    pl.col("trip_end").is_not_null()
)
print(f"Number of rows that will be affected: {data.filter(fill_condition_seconds).shape[0]}")
data = data.with_columns(
    pl.when(fill_condition_seconds)
    .then(pl.col("trip_end") - pl.col("trip_start"))
    .otherwise(pl.col("trip_seconds"))
    .alias("trip_seconds")
)

In [ ]:
missing_counts = data.select([
    pl.col(col).is_null().sum().alias(col) for col in data.columns
])
print("Missing values by column:")
display(missing_counts)

## 3.2) Missing location values
Location values are all latitude, longitude, census tracts, locations, and community areas. 

It was first tried to interpolate the values based on other trips data, so if either census or coordinations were given to interpolate based on the other records values but this seemed to be difficult this nearly interpolated none of the records. We assume that this is due to most of the trips missing data comes from the privacy ensurance of the data. This means similar trips have the same missing information.

Additional methods to interpolate may also be not worth the work as the number of rows which can be interpolated is quite low. Most of the cases if one is missing, all is missing.

Therefore, we decided to drop the rows which do not have pickup locations available. The destination/dropoff is not as relevant because we want to predict the demand of taxis in an area, so how many taxis are estimated to start a trip from the area and not where the trip ends.

### 3.2.1) Latitude and longitude

In [ ]:
# number of rows with missing lat/lon but valid census tract
mask_pu = (
    pl.col("pickup_lat").is_null() &
    pl.col("pickup_lon").is_null() &
    pl.col("pickup_census_tract").is_not_null()
)
mask_do = (
    pl.col("dropoff_lat").is_null() &
    pl.col("dropoff_lon").is_null() &
    pl.col("dropoff_census_tract").is_not_null()
)
print(f"Rows with missing pickup lat/lon but valid census tract: {data.filter(mask_pu).shape[0]}")
print(f"Rows with missing dropoff lat/lon but valid census tract: {data.filter(mask_do).shape[0]}")

In [ ]:
missing_counts = data.select([
    pl.col(col).is_null().sum().alias(col) for col in ['pickup_census_tract', 'pickup_lat', 'pickup_lon', 
                                                       'dropoff_census_tract', 'dropoff_lat', 'dropoff_lon', 
                                                       'pickup_location', 'dropoff_location', 'pickup_community_area', 'dropoff_community_area']
])
print("Missing values by column:")
display(missing_counts)

In [ ]:
data = data.drop_nulls(subset=["pickup_lat", "pickup_lon"])
statistics.append(("Removed rows with missing lat/lon", original_rows - data.shape[0]))

In [ ]:
statistics

In [ ]:
df = data.to_pandas()

In [ ]:
del data

### 3.2.2) Census Tracts
Latitude and longitude coordinations are more precise than census tracts. Still for the spatial analysis these are relevant and offer additional information. As many are missing (see below) they are imputed using a dataset from US Government about census tracts (https://www.census.gov/cgi-bin/geo/shapefiles/index.php?year=2024&layergroup=Census+Tracts)

In [ ]:
# number of records with missing census tract but with valid lat/lon
mask_pu = (
    df['pickup_lat'].notna() &
    df['pickup_lon'].notna() &
    df['pickup_census_tract'].isna()
)
mask_do = (
    df['dropoff_lat'].notna() &
    df['dropoff_lon'].notna() &
    df['dropoff_census_tract'].isna()
)
print(f"After imputation - Rows with missing census tract but valid pickup lat/lon: {mask_pu.sum()}")
print(f"After imputation - Rows with missing census tract but valid dropoff lat/lon: {mask_do.sum()}")

In [ ]:
df.head(1)

In [ ]:
pickup_remaining = (
    df["pickup_census_tract"].isna() & df["pickup_lat"].notna() & df["pickup_lon"].notna()
).sum()
dropoff_remaining = (
    df["dropoff_census_tract"].isna() & df["dropoff_lat"].notna() & df["dropoff_lon"].notna()
).sum()

print("Missing census tracts after spatial fill:")
print(f"  pickup:  {pickup_remaining:,}")
print(f"  dropoff: {dropoff_remaining:,}")

In [ ]:
# Prompt: I want to fill the missing census tracts to ensure by using the shp file. Using the available latitude and longitude from the trips, either pickup and dropoff to fill the missing dropoff_census_tract and pickup_census_tract.
census_gdf = gpd.read_file("../data/tl_2024_17_tract/tl_2024_17_tract.shp")
import geopandas as gpd

# Build a tract lookup from the census shapefile that is already loaded above.
tract_col_candidates = ["GEOID", "geoid", "GEOID20", "TRACTCE", "tractce"]
tract_col = next((col for col in tract_col_candidates if col in census_gdf.columns), None)
if tract_col is None:
    raise KeyError(f"Could not find a census tract id column in census_gdf: {list(census_gdf.columns)}")

tract_gdf = census_gdf[[tract_col, "geometry"]].copy().to_crs(epsg=4326)
tract_gdf[tract_col] = pd.to_numeric(tract_gdf[tract_col], errors="coerce").astype("Int64")


def fill_missing_tracts(frame, target_col, lat_col, lon_col):
    mask = frame[target_col].isna() & frame[lat_col].notna() & frame[lon_col].notna()
    if not mask.any():
        return 0

    points = gpd.GeoDataFrame(
        frame.loc[mask].copy(),
        geometry=gpd.points_from_xy(frame.loc[mask, lon_col], frame.loc[mask, lat_col]),
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(points, tract_gdf[[tract_col, "geometry"]], how="left", predicate="within")
    frame.loc[mask, target_col] = joined[tract_col].to_numpy()
    return int(mask.sum())


pickup_filled = fill_missing_tracts(df, "pickup_census_tract", "pickup_lat", "pickup_lon")
dropoff_filled = fill_missing_tracts(df, "dropoff_census_tract", "dropoff_lat", "dropoff_lon")

print(f"Filled pickup census tracts from coordinates: {pickup_filled:,}")
print(f"Filled dropoff census tracts from coordinates: {dropoff_filled:,}")
print(
    "Remaining missing pickup tracts with coordinates: "
    f"{((df['pickup_census_tract'].isna()) & df['pickup_lat'].notna() & df['pickup_lon'].notna()).sum():,}"
)
print(
    "Remaining missing dropoff tracts with coordinates: "
    f"{((df['dropoff_census_tract'].isna()) & df['dropoff_lat'].notna() & df['dropoff_lon'].notna()).sum():,}"
)

df.head(3)

In [ ]:
pickup_remaining = (
    df["pickup_census_tract"].isna() & df["pickup_lat"].notna() & df["pickup_lon"].notna()
).sum()
dropoff_remaining = (
    df["dropoff_census_tract"].isna() & df["dropoff_lat"].notna() & df["dropoff_lon"].notna()
).sum()

print("Missing census tracts after spatial fill:")
print(f"  pickup:  {pickup_remaining:,}")
print(f"  dropoff: {dropoff_remaining:,}")

### 3.2.3) Trip distance

In [ ]:
# convert to polars
data = pl.from_pandas(df)
del df
data.head(3)

In [ ]:
missing_counts = data.select([
    pl.col(col).is_null().sum().alias(col) for col in data.columns
])
print("Missing values by column:")
display(missing_counts)

# 4) Duplicates
One taxi can only drive one trip at a time, so if combination of a taxi_id and a trip_start returns more than one record it is a duplicate.
As we cannot say in general which of these trips is correct, we drop both.

In [ ]:
# check for duplicates in trip_id
before = len(data)
data = data.unique(subset=["taxi_id", 'trip_start'], keep="none")
dups = before - len(data)
print(f"Total rows: {before:,}, Unique trip_ids: {len(data):,}, Duplicates: {dups:,}")
statistics.append(("Removed duplicate trip_id rows", dups))

In [ ]:
data = data.filter(
    (pl.col("tips_usd") < 0)
)

# 5) Ilogical data
Remove if **any** of:
- `trip_seconds < 60` (under 1 minute)
- `trip_miles < 0.01` (under 0.01 miles)
- `fare_usd == 0` (zero fare)
- trip_seconds, trip_miles, fare_usd, tips_usd, tolls_usd, extra_usd, trip_total_usd is negative
- `average_speed_mph > 100`
- 

All removed rows go into `Removed_Rows.parquet` with a reason.

In [ ]:
length = len(data)

data = data.filter(
    (pl.col("trip_seconds") < 0)
)
print(f"Rows with negative trip_seconds removed: {length - len(data):,}")

data = data.filter(
    (pl.col("trip_miles") < 0)
)
print(f"Rows with negative trip_miles removed: {length - len(data):,}")

data = data.filter(
    (pl.col("fare_usd") < 0)
)
print(f"Rows with negative fare_usd removed: {length - len(data):,}")

data = data.filter(
    (pl.col("tips_usd") < 0)
)
print(f"Rows with negative tips_usd removed: {length - len(data):,}")

data = data.filter(
    (pl.col("tolls_usd") < 0)
)
print(f"Rows with negative tolls_usd removed: {length - len(data):,}")

data = data.filter(
    (pl.col("extras_usd") < 0)
)
print(f"Rows with negative extras_usd removed: {length - len(data):,}")

data = data.filter(
    (pl.col("trip_total_usd") < 0)
)
print(f"Rows with negative trip_total_usd removed: {length - len(data):,}")
# check for negative values in key numeric columns and remove them

print(f"Summary: Removed {length - len(data):,} rows with negative values.")
statistics.append(("Removed rows with negative values", length - len(data)))
print(f"\nRemaining after invalid trip removal: {len(data):,}")

In [ ]:

data = data.filter(
    ~(
        (pl.col("trip_seconds") < 30) |
        (pl.col("trip_miles") < 0.01) |
        (pl.col("fare_usd") == 0)
    )
)


data = data.with_columns(
    (pl.col('trip_miles') / (pl.col('trip_seconds') / 3600)).alias('speed_mph')
).filter(pl.col('speed_mph') > 100)

print(f"Removed {length - len(data):,} rows with unrealistic speeds.")
statistics.append(("Removed rows with unrealistic speeds", length - len(data)))

print(f"Removed {length - len(data):,} rows with invalid trip characteristics.")
statistics.append(("Removed rows with invalid trip characteristics", length - len(data)))

print(f"\nRemaining after invalid trip removal: {len(data):,}")

In [ ]:
display(data.head(3))

# x) Column cleanup
## x.1) Column: Payment Type

In [ ]:
data = data.with_columns(
    pl.col("payment_type")
    .str.to_lowercase()
    .str.strip_chars()
    .str.replace_all(r"\s+", " ", literal=False)
)

# 2. Print unique count
print(f"Payment types: {data['payment_type'].n_unique()}")

# 3. Print value counts
vc = data["payment_type"].value_counts()
display(vc)

fig, ax = plt.subplots(figsize=(10, 6))
sns.set_style("whitegrid")
ax.pie(vc["count"], labels=vc["payment_type"], autopct='%1.1f%%')
plt.title("Distribution of Payment Types")
plt.show()

## x.2) Column: Company 

In [ ]:
data = data.with_columns(
    pl.col("company")
    .str.to_lowercase()
    .str.strip_chars()
    .str.replace_all(r"\s+", " ", literal=False)
    .str.replace_all(r"[,\.]", "", literal=False)
)

# 2. Map the manual replacements
manual_mappings = {
    "choice taxi association inc": "choice taxi association",
    "blue ribbon taxi association inc": "blue ribbon taxi association",
    "top cab affiliation": "top cab",
    "medallion leasin": "medallion leasing",
}
data = data.with_columns(
    pl.col("company").replace(manual_mappings)
)

print(f"Unique companies: {data['company'].n_unique()}")

vc = data["company"].value_counts().sort("count", descending=True)
display(vc)

fig, ax = plt.subplots(figsize=(10, 6))
sns.set_style("whitegrid")
ax.bar(vc['company'], vc["count"])
plt.xticks(rotation=90, ha='right')
plt.title("Distribution of Companies")
plt.show()

---
## 6) DROP ROWS WITH NO SPATIAL DATA
Remove rows where ALL spatial columns are null. Save to separate parquet.

In [ ]:
# spatial_cols = [
#     "pickup_census_tract", "dropoff_census_tract",
#     "pickup_community_area", "dropoff_community_area",
#     "pickup_lat", "pickup_lon", "pickup_location",
#     "dropoff_lat", "dropoff_lon", "dropoff_location",
# ]

# no_spatial_mask = df[spatial_cols].isna().all(axis=1)
# no_spatial_count = no_spatial_mask.sum()
# df[no_spatial_mask].to_parquet(str(OUTPUT_NO_SPATIAL), index=False)
# df = df[~no_spatial_mask].reset_index(drop=True)

# print(f"Removed {no_spatial_count:,} rows with no spatial data at all")
# print(f"Remaining: {len(df):,} rows")

---
## 11) CONSISTENCY & LOGIC CHECKS

---
## 12) ABSOLUTE OUTLIER FILTERING
Hard cutoffs:
- `trip_miles > 200` -> remove
- `trip_seconds > 7200` (> 2h) -> remove
- `fare_usd < 3.25` (below basefare) -> remove
- No upper cut on fare

In [ ]:
df = flag_and_remove(
    df,
    (df["trip_miles"].notna()) & (df["trip_miles"] > 200),
    "trip_miles_over_200",
)

df = flag_and_remove(
    df,
    (df["trip_seconds"].notna()) & (df["trip_seconds"] > 7200),
    "trip_seconds_over_2h",
)

df = flag_and_remove(
    df,
    (df["fare_usd"].notna()) & (df["fare_usd"] < 3.25),
    "fare_below_basefare_3.25",
)

print(f"\nRemaining after absolute outlier filtering: {len(df):,}")

---
## 13) DROP LOCATION TEXT COLUMNS & SAVE CLEANED PARQUET
Keep census tract columns (needed later). Drop only the point-location text columns.

In [ ]:
cols_to_drop = ["pickup_location", "dropoff_location"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Final columns: {df.columns.tolist()}")
print(f"Final rows: {len(df):,}")

df.to_parquet(str(OUTPUT_CLEAN), index=False)
print(f"Saved {OUTPUT_CLEAN}")

---
## 14) ID REMAPPING + COMPACT PARQUET

In [ ]:
trip_id_map = pd.DataFrame({
    "trip_id_int": range(1, len(df) + 1),
    "trip_id_orig": df["trip_id"].values,
})

taxi_ids_unique = df["taxi_id"].dropna().unique()
taxi_id_map = pd.DataFrame({
    "taxi_id_int": range(1, len(taxi_ids_unique) + 1),
    "taxi_id_orig": taxi_ids_unique,
})
taxi_lookup = dict(zip(taxi_id_map["taxi_id_orig"], taxi_id_map["taxi_id_int"]))

df["trip_id_int"] = range(1, len(df) + 1)
df["taxi_id_int"] = df["taxi_id"].map(taxi_lookup)

df.drop(columns=["trip_id", "taxi_id"], inplace=True)

cols_order = ["trip_id_int", "taxi_id_int"] + [c for c in df.columns if c not in ("trip_id_int", "taxi_id_int")]
df = df[cols_order]

trip_id_map.to_csv(str(OUTPUT_TRIP_ID_MAP), index=False)
taxi_id_map.to_csv(str(OUTPUT_TAXI_ID_MAP), index=False)

print(f"trip_id_map: {len(trip_id_map):,} rows")
print(f"taxi_id_map: {len(taxi_id_map):,} rows")

---
## 15) TIME FEATURES FROM trip_start

In [ ]:
ts = df["trip_start"]

df["hour"] = ts.dt.hour.astype("int8")
df["day_of_week"] = ts.dt.dayofweek.astype("int8")
df["month"] = ts.dt.month.astype("int8")
df["date"] = ts.dt.date
df["is_weekend"] = df["day_of_week"].isin([5, 6])

df["bin_30min"] = ts.dt.floor("30min")
df["bin_1h"] = ts.dt.floor("1h")
df["bin_4h"] = ts.dt.floor("4h")
df["bin_1d"] = ts.dt.normalize()
df["bin_1w"] = ts.dt.to_period("W").apply(lambda p: p.start_time)

time_cols = ["hour", "day_of_week", "month", "date", "is_weekend",
            "bin_30min", "bin_1h", "bin_4h", "bin_1d", "bin_1w"]

print(f"Added {len(time_cols)} time columns.")
df[time_cols].sample(10, random_state=42)

---
## 16) CYCLIC ENCODING (sin/cos)

In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

cyclic_cols = ["hour", "hour_sin", "hour_cos",
              "day_of_week", "dow_sin", "dow_cos",
              "month", "month_sin", "month_cos"]

print(f"Added 6 cyclic encoding columns.")
df[cyclic_cols].sample(10, random_state=42)

---
## 17) ELECTRONIC PAYMENT FEE DETECTION + SAVE COMPACT PARQUET

In [ ]:
df["has_electronic_fee"] = (
    df["payment_type"].isin(["credit card", "mobile"])
    & ((df["trip_total_usd"] - df[["fare_usd","tips_usd","tolls_usd","extras_usd"]].sum(axis=1)).round(2) > 0.49)
    & ((df["trip_total_usd"] - df[["fare_usd","tips_usd","tolls_usd","extras_usd"]].sum(axis=1)).round(2) < 0.51)
)

df.to_parquet(str(OUTPUT_COMPACT), index=False)
print(f"Saved compact parquet: {OUTPUT_COMPACT}")
print(f"has_electronic_fee: {df['has_electronic_fee'].sum():,} rows")

---
## 18) SAVE REMOVED ROWS

In [ ]:
print(original_rows)

In [ ]:
if removed_frames:
    removed_df = pd.concat(removed_frames, ignore_index=True)
    removed_df.to_parquet(str(OUTPUT_REMOVED), index=False)
    print(f"Saved {len(removed_df):,} removed rows to {OUTPUT_REMOVED}")
    print(f"\nRemoval breakdown:")
    print(removed_df["removal_reason"].value_counts().to_string())
else:
    print("No rows were removed.")

print(f"\n=== CLEANING SUMMARY ===")
print(f"Original rows:  {original_rows:,}")
print(f"Final rows:     {len(df):,}")
removed_total = original_rows - len(df)
print(f"Removed:        {removed_total:,} ({removed_total/original_rows*100:.2f}%)")